In [ ]:
import pandas as pd
from rouge_score import rouge_scorer
import sacrebleu
import evaluate
import bert_score

# Load your CSV
df = pd.read_csv("predicted_vs_actual_gemma_2500.csv")  # or your modified file

# Get references and predictions
references = df['Actual'].astype(str).tolist()
predictions = df['Predicted'].astype(str).tolist()

### ROUGE
rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge1, rouge2, rougeL = 0, 0, 0
for ref, pred in zip(references, predictions):
    scores = rouge.score(ref, pred)
    rouge1 += scores['rouge1'].fmeasure
    rouge2 += scores['rouge2'].fmeasure
    rougeL += scores['rougeL'].fmeasure
n = len(references)
print(f"ROUGE-1: {rouge1/n:.4f}")
print(f"ROUGE-2: {rouge2/n:.4f}")
print(f"ROUGE-L: {rougeL/n:.4f}")

### BLEU 1,2,3
bleu1 = sacrebleu.corpus_bleu(predictions, [references], smooth_method="exp", smooth_value=0.0, force=True, lowercase=True, tokenize="intl", use_effective_order=False)
print(f"BLEU-1: {bleu1.score:.4f}")

bleu2 = evaluate.load("bleu")
results_bleu2 = bleu2.compute(predictions=predictions, references=[[ref] for ref in references], max_order=2)
print(f"BLEU-2: {results_bleu2['bleu']:.4f}")

results_bleu3 = bleu2.compute(predictions=predictions, references=[[ref] for ref in references], max_order=3)
print(f"BLEU-3: {results_bleu3['bleu']:.4f}")

### METEOR
meteor = evaluate.load("meteor")
results_meteor = meteor.compute(predictions=predictions, references=references)
print(f"METEOR: {results_meteor['meteor']:.4f}")

### BERTScore
P, R, F1 = bert_score.score(predictions, references, lang="en", verbose=True)
print(f"BERTScore F1: {F1.mean().item():.4f}")
